# 06-4. 메시지 경계와 프로토콜 설계 예제

## Goal

- 길이 접두사 프레임을 구성합니다.
- 완전한 프레임과 남은 바이트를 분리합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

네트워크 바이트 순서의 4바이트 길이 필드를 사용합니다.


## Steps

### 길이 접두사 인코딩과 디코딩

TCP 바이트 흐름에서 메시지 경계를 직접 표현합니다.


In [1]:
import struct

HEADER_SIZE = 4
MAX_PAYLOAD = 1024


def encode_frame(text: str) -> bytes:
    payload = text.encode("utf-8")
    if len(payload) > MAX_PAYLOAD:
        raise ValueError("payload가 너무 큽니다")
    return struct.pack("!I", len(payload)) + payload


def decode_one(buffer: bytes):
    if len(buffer) < HEADER_SIZE:
        return None, buffer
    size = struct.unpack("!I", buffer[:HEADER_SIZE])[0]
    if size > MAX_PAYLOAD:
        raise ValueError("선언한 payload 크기가 제한을 초과했습니다")
    end = HEADER_SIZE + size
    if len(buffer) < end:
        return None, buffer
    return buffer[HEADER_SIZE:end].decode("utf-8"), buffer[end:]


stream = encode_frame("첫째") + encode_frame("둘째")
first, remainder = decode_one(stream)
second, remainder = decode_one(remainder)
print(first, second, "남은 바이트:", len(remainder))


첫째 둘째 남은 바이트: 0


## Checks

한글의 문자 수와 UTF-8 바이트 수가 다름을 함께 확인합니다.


In [2]:
assert first == "첫째" and second == "둘째"
assert remainder == b""
partial = encode_frame("경계")[:-1]
decoded, saved = decode_one(partial)
assert decoded is None and saved == partial
assert len("경계") != len("경계".encode("utf-8"))
print("프레임 검사 통과")


프레임 검사 통과


## Next Steps

프레임 크기 제한은 메모리 고갈을 막기 위해 헤더를 읽은 직후 적용합니다.
